# 1.0 - Exploracion de los viajes de MiBici (EDA)

**Objetivo:** entender volumen, calidad y patrones de los datos para justificar las decisiones
de la propuesta: rango de fechas, filtros de limpieza, granularidad, etiqueta de P2 y
variables de P3.

> Este notebook es **solo de exploracion**. Lo que resulto util ya vive en el modulo:
> filtros y variables en `features.py`, graficas en `plots.py` y parametros en `config.py`.
> Requiere haber corrido la ingesta y features (ver README). Antes de subirlo, reinicia el
> kernel y ejecuta *Run All* para que las salidas correspondan al codigo.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

from tc5061_proyecto_equipo18 import config, plots
from tc5061_proyecto_equipo18.dataset import Manifiesto
from tc5061_proyecto_equipo18.features import (
    agregar_bloques,
    cargar_estaciones,
    cargar_viajes,
    filtrar_viajes_validos,
    perfiles_estacion,
)

manifiesto = Manifiesto.cargar()
print(
    f"Datos version {manifiesto.version}: {manifiesto.mes_inicio} a {manifiesto.mes_fin}, "
    f"{manifiesto.total_viajes:,} viajes"
)

## 1. Volumen por mes

**Que buscar:** meses atipicos (vacaciones, lluvias, eventos) y si el volumen es estable a lo
largo del rango elegido.

In [ ]:
viajes = cargar_viajes()
estaciones = cargar_estaciones()
fig = plots.grafica_viajes_por_mes(viajes)

## 2. Calidad: duracion de los viajes y estaciones desconocidas

**Que buscar:** cuantos viajes duran menos de 1 minuto (desanclajes fallidos) o mas de 4 horas
(bicicletas mal ancladas), y cuantos usan estaciones que no estan en la nomenclatura. Estos
cortes son `config.DURACION_MIN_SEG` y `config.DURACION_MAX_SEG`.

In [ ]:
duracion_min = viajes["duracion_seg"] / 60
print(duracion_min.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(1))
print(
    f"< 1 min: {(viajes['duracion_seg'] < 60).mean():.2%} - "
    f"> 4 h: {(viajes['duracion_seg'] > 4 * 3600).mean():.2%}"
)
validos, filtros = filtrar_viajes_validos(viajes, estaciones)
pd.Series(filtros)

**Hallazgos (completar al ejecutar):**

- ...

## 3. Patrones temporales

**Que buscar:** picos de entrada y salida en dias laborales y un patron distinto en fines de
semana; justifican la granularidad horaria de P1 y las variables de calendario.

In [ ]:
conteos = pd.read_parquet(config.CONTEOS_HORA_PARQUET)
fig = plots.grafica_perfil_horario(conteos)

In [ ]:
por_dia = conteos.assign(dia=conteos["hora"].dt.normalize()).groupby("dia")["salidas"].sum()
nombres = dict(enumerate(["lun", "mar", "mie", "jue", "vie", "sab", "dom"]))
por_dia.groupby(por_dia.index.dayofweek).mean().rename(index=nombres).round(0)

## 4. Estaciones

**Que buscar:** como se reparte la demanda entre estaciones (unas pocas concentran mucho
volumen) y la distribucion de capacidades.

In [ ]:
volumen = validos.groupby("origen_id").size()
fig = plots.grafica_mapa_estaciones(estaciones, volumen)
print(estaciones["capacidad"].describe().round(1))
(
    estaciones.assign(viajes=estaciones["estacion_id"].map(volumen))
    .nlargest(10, "viajes")[["codigo", "nombre", "municipio", "capacidad", "viajes"]]
)

## 5. P2: flujo neto por bloque frente a la capacidad

**Que buscar:** que proporcion de bloques quedaria en riesgo con distintos umbrales. Un umbral
muy bajo marca casi todo como riesgo; uno muy alto casi nada. El valor elegido esta en
`config.P2_UMBRAL_RIESGO`.

In [ ]:
bloques = agregar_bloques(conteos, config.P2_HORAS_BLOQUE).merge(
    estaciones[["estacion_id", "capacidad"]], on="estacion_id"
)
bloques = bloques[bloques["capacidad"] > 0]
relativo = bloques["neto"] / bloques["capacidad"]
for umbral in (0.3, 0.4, 0.5, 0.6, 0.75):
    print(
        f"umbral {umbral:.2f}: vaciado {(relativo >= umbral).mean():.2%} - "
        f"llenado {(-relativo >= umbral).mean():.2%}"
    )

## 6. P3: perfiles de uso de dos estaciones contrastantes

**Que buscar:** estaciones de origen matutino (salen bicicletas en la manana y regresan en la
tarde) frente a estaciones de destino laboral (al reves). Esa diferencia es la que el
clustering debe capturar.

In [ ]:
fin_entrenamiento = pd.Timestamp(config.FIN_ENTRENAMIENTO) + pd.Timedelta(days=1)
perfiles = perfiles_estacion(conteos, conteos["hora"].min(), fin_entrenamiento)
print(f"{len(perfiles)} estaciones con perfil en el periodo de entrenamiento")
contrastantes = pd.concat([perfiles.nsmallest(1, "balance_neto"), perfiles.nlargest(1, "balance_neto")])

horas = range(24)
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, (_, fila) in zip(ejes, contrastantes.iterrows()):
    ax.plot(horas, [fila[f"sal_lab_h{h:02d}"] * 100 for h in horas], color=plots.SERIES[0], label="Salidas")
    ax.plot(horas, [fila[f"lle_lab_h{h:02d}"] * 100 for h in horas], color=plots.SERIES[1], label="Llegadas")
    ax.set_title(f"Estacion {int(fila['estacion_id'])} - balance neto {fila['balance_neto']:+.2f}")
    ax.set_xlabel("Hora del dia")
ejes[0].set_ylabel("% de los viajes del dia (laboral)")
ejes[0].legend()

## Conclusiones y decisiones trasladadas al modulo

| Hallazgo de la exploracion | Decision | Donde quedo |
|---|---|---|
| Viajes de menos de 1 minuto o mas de 4 horas | Se filtran antes de agregar | `features.filtrar_viajes_validos`, `config.DURACION_*` |
| Ciclos diario y semanal muy marcados | Granularidad horaria, rezagos de 1, 2, 7 y 14 dias, calendario y festivos | `features.construir_dataset_p1` |
| Flujos netos grandes respecto a la capacidad en ciertos bloques | Etiqueta de P2 con umbral relativo a la capacidad, bloques de 3 h | `features.etiquetar_riesgo`, `config.P2_*` |
| Estaciones de origen y de destino con perfiles opuestos | Perfiles horarios separados para salidas y llegadas, laboral y fin de semana | `features.perfiles_estacion` |

**Hallazgos adicionales (completar al ejecutar):**

- ...